# Managing Errors and Exceptions: Handling Messy Order Data

In this practice you will use `try`, `except`, `else` and `finally` to handle errors that real order data causes. You will also raise your own error when a value is invalid, and handle a missing file.

## Background

Orders sometimes arrive from a web form or a spreadsheet with problems: a quantity typed as words, a quantity of zero, or a file that has not been exported yet. Without error handling, one bad record stops the whole program.

## Data Description

All the practices in this module use **Retail Orders.csv**: 200 orders from 2025 placed with a small online retailer based in Ireland. The data is fictional and was created for teaching.

| Column Name | Description |
|---|---|
| order_id | Unique order identifier |
| order_date | Date the order was placed (YYYY-MM-DD) |
| delivery_date | Date the order was delivered; blank if not yet delivered |
| customer_name | Customer's name, as typed into the order system |
| city | Delivery city |
| product | Product ordered |
| category | Product category: Electronics, Office or Home |
| quantity | Number of units ordered |
| unit_price | Price of one unit, in euro |
| priority | Order priority: Urgent, Normal or Minor |
| status | Order status: Delivered, Shipped or Cancelled |

## 1. Load the Dataset

We import pandas and read the CSV file into a DataFrame. Keep **Retail Orders.csv** in the same folder as this notebook.

In [1]:
import pandas as pd

df = pd.read_csv("Retail Orders.csv")
df.head()

,order_id,order_date,delivery_date,customer_name,city,product,category,quantity,unit_price,priority,status
0,ORD-1001,2025-01-11,2025-01-19,Tadhg Farrell,Limerick,Filing Cabinet,Office,5,120.0,Minor,Delivered
1,ORD-1002,2025-01-11,2025-01-15,Liam Kelly,Dublin,Whiteboard,Office,3,45.0,Normal,Delivered
2,ORD-1003,2025-01-11,2025-01-13,Sean Ryan,Cork,A4 Notebook,Office,3,4.5,Normal,Delivered
3,ORD-1004,2025-01-12,2025-01-16,Sinead Ryan,Cork,Ergonomic Chair,Office,5,189.0,Normal,Delivered
4,ORD-1005,2025-01-15,2025-01-24,Tomas Lynch,Galway,A4 Notebook,Office,1,4.5,Minor,Delivered


## 2. An Unhandled Error

The average price per item is the total divided by the quantity. If the quantity is zero, Python stops with a `ZeroDivisionError`.

The next cell is meant to fail. If you use *Run All*, Jupyter stops here, so run the cells after it one at a time.

In [2]:
def price_per_item(total, quantity):
    return total / quantity

price_per_item(135.0, 0)

ZeroDivisionError: float division by zero

## 3. Handle It with try-except

The `try` block runs the code. If a `ZeroDivisionError` happens, the `except` block runs instead and the program carries on.

In [3]:
def price_per_item(total, quantity):
    try:
        return total / quantity
    except ZeroDivisionError:
        return "Error: quantity is zero."

print(price_per_item(135.0, 3))
print(price_per_item(135.0, 0))

45.0
Error: quantity is zero.


## 4. Handle a ValueError

`int()` cannot convert text such as `"three"`, so it raises a `ValueError`.

In [4]:
def read_quantity(text):
    try:
        return int(text)
    except ValueError:
        return "Error: not a valid quantity."

print(read_quantity("3"))
print(read_quantity("three"))

3
Error: not a valid quantity.


## 5. Handle Several Exceptions

Here are four raw records as they might arrive from a web form. Each one is processed in a single `try` block with a separate `except` for each kind of problem.

In [5]:
raw_orders = [
    ("ORD-9001", "135.00", "3"),
    ("ORD-9002", "89.50", "two"),
    ("ORD-9003", "60.00", "0"),
    ("ORD-9004", "240.00", "4"),
]

for order_id, total_text, qty_text in raw_orders:
    try:
        per_item = float(total_text) / int(qty_text)
        print(order_id, "price per item:", per_item)
    except ValueError:
        print(order_id, "Error: quantity is not a whole number.")
    except ZeroDivisionError:
        print(order_id, "Error: quantity is zero.")

ORD-9001 price per item: 45.0
ORD-9002 Error: quantity is not a whole number.
ORD-9003 Error: quantity is zero.
ORD-9004 price per item: 60.0


## 6. Catch Any Exception

`except Exception as e` catches almost any runtime error and stores its message in `e`. It is useful as a last resort, but specific exceptions are better because they say exactly what went wrong.

In [6]:
orders = df.head(3).to_dict("records")

try:
    print(orders[5]["product"])
except Exception as e:
    print("Error:", e)

Error: list index out of range


## 7. The else and finally Clauses

`else` runs only if no error happened. `finally` always runs, error or not, so it suits clean-up messages.

In [7]:
def check_order(total, quantity):
    try:
        per_item = total / quantity
    except ZeroDivisionError:
        print("Cannot work out price per item.")
    else:
        print("Price per item:", per_item)
    finally:
        print("Check complete.")

check_order(945.0, 5)
check_order(945.0, 0)

Price per item: 189.0
Check complete.
Cannot work out price per item.
Check complete.


## 8. Raise Your Own Exception

A negative quantity is not a Python error, but it is a business error. `raise ValueError(...)` stops the function with a clear message, which the caller then handles.

In [8]:
def validate_quantity(quantity):
    if quantity < 0:
        raise ValueError("Quantity cannot be negative.")
    return "Quantity " + str(quantity) + " is valid."

for q in [3, -2]:
    try:
        print(validate_quantity(q))
    except ValueError as e:
        print("ValueError:", e)

Quantity 3 is valid.
ValueError: Quantity cannot be negative.


## 9. Handle a Missing File

The 2024 orders file has not been exported yet. Opening it raises a `FileNotFoundError`, which we catch with a helpful message.

In [9]:
try:
    with open("Retail Orders 2024.csv", "r") as file:
        content = file.read()
except FileNotFoundError:
    print("File not found: Retail Orders 2024.csv has not been exported yet.")

File not found: Retail Orders 2024.csv has not been exported yet.


## 10. Other Common Exception Types

Three more exceptions you will meet often with order data: a `TypeError` from mixing text and numbers, an `IndexError` from a position that does not exist in a list, and a `KeyError` from a key that does not exist in a dictionary.

In [10]:
try:
    print("Total: " + 135.0)
except TypeError as e:
    print("TypeError:", e)

try:
    quantities = [3, 3, 5]
    print(quantities[5])
except IndexError as e:
    print("IndexError:", e)

try:
    order = {"order_id": "ORD-1004", "city": "Cork"}
    print(order["country"])
except KeyError as e:
    print("KeyError:", e)

TypeError: can only concatenate str (not "float") to str
IndexError: list index out of range
KeyError: 'country'


## Summary

In this practice you:

- saw how an unhandled error stops a program
- handled `ZeroDivisionError` and `ValueError` with `try-except`
- used several `except` blocks, and `except Exception as e` as a last resort
- used `else` for code that runs only when there is no error, and `finally` for code that always runs
- raised your own `ValueError` for an invalid quantity
- handled a missing file with `FileNotFoundError`

In Lesson 8 you will use decorators to add checks like these to functions without changing them.